In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from scipy.stats import norm
import os

from model import *
from data import generate_gmm_data
from train_utils import train

from plot import *
from utils import *


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
relu = lambda X: np.maximum(X, 0)
d_relu = lambda X: np.maximum(np.sign(X), 0)

elu = lambda X, alpha=1.0: ( alpha * (np.exp(X) - 1) ) * (X <= 0) + (X * (X > 0))
d_elu = lambda X, alpha=1.0: ( alpha * np.exp(X) ) * (X <= 0) + (X > 0)

gelu = lambda X: X * norm.cdf(X)
d_gelu = lambda X: norm.cdf(X) + X * norm.pdf(X)

sigmoid = lambda X: 1 / (1 + np.exp(-X))
d_sigmoid = lambda X: sigmoid(X) * (1 - sigmoid(X))

silu = lambda X: X * sigmoid(X)
d_silu = lambda X: sigmoid(X) + X * d_sigmoid(X)

leaky_relu = lambda X, slope=0.01: slope * np.minimum(X, 0) + np.maximum(X, 0)
d_leaky_relu = lambda X, slope=0.01: slope * (X <= 0) + (X > 0).astype(float)

identity = lambda X: X
d_identity = lambda X: np.ones(X.shape)

abs_ = lambda X: np.abs(X)
d_abs_ = lambda X: -1 * (X <= 0) + (X > 0)

softplus = lambda X: np.log(1 + np.exp(X))
d_softplus = lambda X: sigmoid(X)
d2_softplus = lambda X: d_sigmoid(X)

leaky_softplus = lambda X, slope=0.01: slope * X + (1 - slope) * (softplus(X) - np.log(2))
d_leaky_softplus = lambda X, slope=0.01: slope + (1 - slope) * (d_softplus(X))
d2_leaky_softplus = lambda X, slope = 0.01: (1 - slope) * d2_softplus(X)


smooth_abs = lambda X, beta = 10: (2 / beta) * softplus(X) - X - ( (2 / beta) * np.log(2) )
d_smooth_abs = lambda X, beta = 10: (2 / beta) * d_softplus(X) - 1

custom_activations_map = {'ReLU': [relu, d_relu],
               'Leaky-Softplus': [leaky_softplus, d_leaky_softplus], 
               'ELU': [elu, d_elu],
               'GELU': [gelu, d_gelu],
               'SiLU': [silu, d_silu],
               'Leaky-ReLU': [leaky_relu, d_leaky_relu],
               'Abs': [abs_, d_abs_],
               'SmoothAbs': [smooth_abs, d_smooth_abs],
               'Identity': [identity, d_identity]}

In [ ]:
optimizer_map = {'SGD': optim.SGD,
                'Adam': optim.Adam}

# Model Training

In [ ]:
def get_data(d: int, 
             K: int, 
             N_k: int, 
             scale: float = 1.0,
             nu: float = 1.0,
             normalize_: bool = False, 
             orthogonalize_: bool = False,
             whiten_: bool = False):
    
    
    N = K * N_k
    
    # d > N --> random Gaussian data
    if d >= N:
        train_set, train_loader, _ = generate_random_data(d, 
                                                       K, 
                                                       N_k, 
                                                       orthogonalize_ = orthogonalize_, 
                                                       normalize_ = normalize_)
        val_loader = train_loader 
    
    # N > d --> Gaussian mixture data
    elif N > d:
        train_set, train_loader, _ = generate_gmm_data(d, 
                                                    K, 
                                                    N_k,
                                                    nu=nu,
                                                    data_scale = scale,
                                                    whiten_ = whiten_, 
                                                    normalize_ = normalize_)
        val_loader = train_loader
    
    
    X = train_set.samples.T
    Y = train_set.targets.T
    
    return train_loader, val_loader, X, Y

In [ ]:
def train_model(num_layers: int,
                data_dim: int, 
                hidden_dim: int, 
                num_classes: int, 
                act_fn_str: str, 
                train_loader, 
                val_loader, 
                epochs, 
                lr, 
                freq,
                loss_fn, 
                optimizer_type='SGD', 
                freeze_last_layer=True, 
                residual=False, 
                batchnorm=False, 
                init_method='orthogonal',
                last_layer_init_method='orthogonal',
                init_scale=1):
    
    
    # Train model
    model = MLP(num_layers, 
                data_dim, 
                hidden_dim, 
                num_classes, 
                act_fn_str, 
                freeze_last_layer=freeze_last_layer,
                diff_init_last_layer=True,
                residual=residual, 
                batchnorm=batchnorm, 
                init_method=init_method, 
                last_layer_init_method=last_layer_init_method,
                init_scale=init_scale)

    model = model.to(device)
    
    # Training
    criterion = loss_map[loss_fn]
    optimizer = optimizer_map[optimizer_type](model.parameters(), lr=lr, weight_decay=0)
    scheduler = None
    #scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs, eta_min=0)
    results = train(model, 
                    criterion, 
                    optimizer, 
                    train_loader, 
                    val_loader, 
                    epochs, 
                    device,
                    scheduler=scheduler, 
                    return_weights=True,
                    return_grads=True,
                    return_outs=True)


    weights = results['all_weights']
    grads = results['all_grads']
    outs = results['all_outs']
    train_losses = results['train_losses']
    
    return weights, grads, outs, train_losses #outs, train_losses

# Weight decompositions during training

In [ ]:
def fro_inner_product(X: np.ndarray, Y: np.ndarray):
    assert X.shape == Y.shape
    
    X_norm = np.linalg.norm(X, ord='fro')
    Y_norm = np.linalg.norm(Y, ord='fro')
    
    return np.trace(X.T @ Y) / (X_norm * Y_norm)

def normalized_dist(X: np.ndarray, Y: np.ndarray, norm_factor: float = 1.0):
    assert X.shape == Y.shape
    
    return np.linalg.norm(X - Y)**2 / norm_factor 

def normalized_norms(X: np.ndarray):
    return normalized_dist(X, np.zeros(X.shape))

def sin_theta_norms(U1: np.ndarray, U2: np.ndarray):
    svals = np.linalg.svd(U1.T @ U2, compute_uv=False)
    angles = np.arccos(np.clip(svals, 0, 1))
    
    return np.linalg.norm(np.sin(angles))

In [ ]:
'''
Helper function to calculate the norm of each component in the weight matrix 
'''
def weight_decomp_norms_dists(weights, grads, K, init_scale, epochs):
    
    # Get "small update subspace"
    W1_0 = weights[0][0]
    d = W1_0.shape[-1]
    G1_0 = grads[0][0]
    
    L0, s0, R0 = np.linalg.svd(G1_0, full_matrices=True)
    R0 = R0.T
    
    L0_1 = L0[:, :K]
    R0_1 = R0[:, :K]
    
    
    U_null, V_null = small_update_subspace(W1_0, G1_0, K, init_scale)
    p = U_null.shape[-1]
    

    # Get "large update subspace"
    U_update = np.linalg.svd(U_null, full_matrices=True)[0][:, p:]
    V_update = np.linalg.svd(V_null, full_matrices=True)[0][:, p:]
    
    
    # Concatenate two subspaces to create orthogonal matrices
    U = np.concatenate((U_update, U_null), axis=1)
    V = np.concatenate((V_update, V_null), axis=1)

    
    # Get initial weight factors
    W1_0 = weights[0][0]
    
    W11_0_tilde = U_update.T @ W1_0 @ V_update
    W12_0_tilde = U_update.T @ W1_0 @ V_null
    W13_0_tilde = U_null.T @ W1_0 @ V_update
    W14_0_tilde = U_null.T @ W1_0 @ V_null

    
    # Calculate weight decomposition norms and distance from initialization
    # for all weights until last-layer classifier 
    norms = np.zeros((epochs - 1, 4))
    dists = np.zeros((epochs - 1, 4))
    
    for t in range(1, epochs):
        # Get weights and resulting decomposition
        W1_t = weights[t][0]

        W11_t_tilde = U_update.T @ W1_t @ V_update
        W12_t_tilde = U_update.T @ W1_t @ V_null
        W13_t_tilde = U_null.T @ W1_t @ V_update
        W14_t_tilde = U_null.T @ W1_t @ V_null
        

        # Calculate relative squared distance of each component
        norm_factor = np.linalg.norm(W1_t - W1_0, ord='fro')**2
        dists[t - 1, 0] = normalized_dist(W11_t_tilde, W11_0_tilde, norm_factor)
        dists[t - 1, 1] = normalized_dist(W12_t_tilde, W12_0_tilde, norm_factor)
        dists[t - 1, 2] = normalized_dist(W13_t_tilde, W13_0_tilde, norm_factor)
        dists[t - 1, 3] = normalized_dist(W14_t_tilde, W14_0_tilde, norm_factor)
        
        
    return norms, dists

In [ ]:
reduction = 'sum'
loss_map = {
    'ce_loss': nn.CrossEntropyLoss(reduction=reduction),
    'mse_loss': nn.MSELoss(reduction=reduction)
}

data_type_map = {
    'gmm': 'GMM', 
    'uos': 'UoS',
    'random': 'Random'
}

In [ ]:
# Data setup
d = 64 # 128 # Ambient dim
m = 72 #192
K = 4 # Number of classes
N_k = 250
N = N_k * K
nu = np.sqrt(3)
scale = 1

# Model setup
L = 2
act_fn_strs = ['ELU', 'GELU', 'SiLU']
residual = False
batchnorm = False
init_method = 'orthogonal'
last_layer_init_method = 'uniform'
init_scale = 1e-2

whiten_ = True
normalize_ = False
orthogonalize_ = False

# Training setup
epochs = 100
lr = 1e-3
freq = 1
optimizer_type = 'SGD'
#loss_fn = 'ce_loss'
loss_fn = 'mse_loss'

save = True


In [ ]:
trials = 10

weight_decomp_norms = np.zeros((trials, len(act_fn_strs), 2, epochs - 1, 4))
weight_decomp_dists = np.zeros((trials, len(act_fn_strs), 2, epochs - 1, 4))


train_losses = np.zeros((trials, len(act_fn_strs), 2, epochs))

nonlin_comps = np.zeros((trials, len(act_fn_strs), 2, epochs))

for i in range(trials):
    for k in range(2):
        freeze_last_layer = (k == 1) # k = 0 --> False --> train W2, k = 1 --> True --> fix W2
        for j, act_fn_str in enumerate(act_fn_strs):
            
            # Create data
            train_loader, val_loader, X, Y = get_data(d, K, N_k, scale, nu, normalize_, orthogonalize_, whiten_)

            print("=== TRIAL " + str(i) + " ===")

            # Train model
            weights, grads, outs, train_losses_trial = train_model(L, d, m, K, act_fn_str, 
                                                             train_loader, 
                                                             val_loader, 
                                                             epochs, 
                                                             lr, 
                                                             freq, 
                                                             loss_fn, 
                                                             optimizer_type=optimizer_type,
                                                             freeze_last_layer=freeze_last_layer,
                                                             residual=residual,
                                                             batchnorm=batchnorm,
                                                             init_method=init_method,
                                                             last_layer_init_method=last_layer_init_method,
                                                             init_scale=init_scale)

                            
            # Plot train loss
            plot_train_losses(train_losses_trial, act_fn_str, d, K, None, savefig=False)
            train_losses[i, j, k, :] = train_losses_trial

            # Calculate norms of decompositions
            trial_decomp_norms, trial_decomp_dists = weight_decomp_norms_dists(weights, grads, K, init_scale, epochs)


            weight_decomp_norms[i, j, k, ...] = trial_decomp_norms
            weight_decomp_dists[i, j, k, ...] = trial_decomp_dists

In [ ]:
#print(np.mean(nonlin_comps, axis=0)[2, 0, :])

In [ ]:
# Mean and stdev of weight decomposition norms across trials
mean_weight_decomp_dists = np.mean(weight_decomp_dists, axis=0)
std_weight_decomp_dists = np.std(weight_decomp_dists, axis=0)

mean_weight_decomp_norms = np.mean(weight_decomp_norms, axis=0)
std_weight_decomp_norms = np.std(weight_decomp_norms, axis=0)

x_range = np.arange(1, epochs)

for k in range(2):
    n_plots = len(act_fn_strs)
    fig, axes = plt.subplots(
        1,
        n_plots,
        figsize=(5 * n_plots, 5),
        sharey=True
    )

    # Make axes iterable even if n_plots == 1
    if n_plots == 1:
        axes = [axes]

    legend_handles = None
    legend_labels = None

    for i, act_fn_str in enumerate(act_fn_strs):
        ax = axes[i]

        for j in range(4):
            label = rf'$\widetilde{{W}}_{{1,{j+1}}}(t)$'

            line, = ax.plot(
                x_range,
                mean_weight_decomp_dists[i, k, :, j],
                linewidth=5,
                label=label
            )

            ax.fill_between(
                x_range,
                mean_weight_decomp_dists[i, k, :, j] - std_weight_decomp_dists[i, k, :, j],
                mean_weight_decomp_dists[i, k, :, j] + std_weight_decomp_dists[i, k, :, j],
                alpha=0.5
            )

            # Capture legend handles only once
            if legend_handles is None:
                legend_handles = []
                legend_labels = []

            if label not in legend_labels:
                legend_handles.append(line)
                legend_labels.append(label)

        ax.set_title(act_fn_str, fontsize=24)
        #ax.set_xlabel('Train Epoch', fontsize=24)
        ax.grid(True)
        ax.tick_params(axis='both', labelsize=18)

        ax.set_yscale('log')

    axes[0].set_ylim([10**-9, 2])
    axes[0].set_ylabel('Norm. Squared \n Dist. from Init.', fontsize=24)

    # Suptitle
    title = '' if whiten_ else 'Unwhitened Data'
    fig.suptitle(title, fontsize=28, y=1.0, x=0.525)
    fig.supxlabel('Train Epoch', fontsize=28, x=0.54)

    # Shared legend below suptitle
    fig.legend(
        legend_handles,
        legend_labels,
        loc='upper center',
        bbox_to_anchor=(0.55, 1.025),
        ncol=4,
        fontsize=24,
        frameon=False
    )

    plt.tight_layout(rect=[0, 0, 1, 0.9])
    
    suptitle = r'Trained $W_2$' if k == 0 else '' #r'Frozen $W_2$'
    plt.suptitle(suptitle, fontsize=28, x=0.55, y=1.1)

    savefig_dir = './figs/verify_theory/'
    os.makedirs(savefig_dir, exist_ok=True)

    #save_fname = 'network_verify_theory_decomp_distances'
    save_fname = 'network_gelu_vs_relu_decomp_distances'
    save_fname += '_trained_W2' if k == 0 else ''
    save_fname += '_unwhitened_data.pdf' if not whiten_ else '.pdf'

    if save:
        plt.savefig(os.path.join(savefig_dir, save_fname), bbox_inches='tight')

    plt.show()
    plt.close()


In [ ]:
# # Plot loss
# mean_train_losses = np.mean(train_losses / N, axis=0)
# std_train_losses = np.std(train_losses / N, axis=0)

# x_range = np.arange(1, epochs + 1)


# n_plots = len(act_fn_strs)
# fig, axes = plt.subplots(
#     1,
#     n_plots,
#     figsize=(5 * n_plots, 5),
#     sharey=True
# )

# # Make axes iterable even if n_plots == 1
# if n_plots == 1:
#     axes = [axes]

# legend_handles = None
# legend_labels = None
# for i, act_fn_str in enumerate(act_fn_strs):
#     ax = axes[i]
#     for k in range(2):
#         label = r'Trained $W_2$' if k == 0 else r'Frozen $W_2$'

#         line, = ax.plot(
#             x_range,
#             mean_train_losses[i, k, :],
#             linewidth=5,
#             label=label
#         )

#         ax.fill_between(
#             x_range,
#             mean_train_losses[i, k, :] - std_train_losses[i, k, :],
#             mean_train_losses[i, k, :] + std_train_losses[i, k, :],
#             linewidth=5,
#             alpha=0.5
#         )

#         # Capture legend handles only once
#         if legend_handles is None:
#             legend_handles = []
#             legend_labels = []

#         if label not in legend_labels:
#             legend_handles.append(line)
#             legend_labels.append(label)

#         ax.set_title(act_fn_str, fontsize=24)
#         ax.grid(True)
#         ax.tick_params(axis='both', labelsize=18)

#     #axes[0].set_ylim([-0.1, 1.1])
#     axes[0].set_ylabel('Mean-Squared Error', fontsize=24)

# # Suptitle
# title = '' if whiten_ else 'Unwhitened Data'
# fig.suptitle(title, fontsize=28, y=1.0, x=0.525)
# fig.supxlabel('Train Epoch', fontsize=28, x=0.54)

# # Shared legend below suptitle
# fig.legend(
#     legend_handles,
#     legend_labels,
#     loc='upper center',
#     bbox_to_anchor=(0.525, 1.025),
#     ncol=4,
#     fontsize=24,
#     frameon=False
# )

# plt.tight_layout(rect=[0, 0, 1, 0.9])

# savefig_dir = './figs/verify_theory/'
# os.makedirs(savefig_dir, exist_ok=True)

# save_fname = 'network_verify_theory_losses'
# save_fname += '_unwhitened_data.pdf' if not whiten_ else '.pdf'

# if save:
#     plt.savefig(os.path.join(savefig_dir, save_fname), bbox_inches='tight')

# plt.show()
# plt.close()
